# PDF → Excel 변환기 (공용차량 운행일지)

**2023-2024년 양식 (폼)** 과 **2025-2026년 양식 (표)** 모두 지원합니다.

셀을 위에서부터 순서대로 `Shift + Enter`로 실행하세요.

## 1단계: 필요한 패키지 설치 (최초 1회만)

In [ ]:
!pip install pdfplumber openpyxl pandas tabula-py

## 2단계: 설정
아래 경로와 옵션을 본인 환경에 맞게 수정하세요.

In [ ]:
#===================================================
# 여기만 수정하세요!
#===================================================

# PDF 파일 또는 폴더 경로
PDF_PATH = r"C:\Users\USER\Documents\txt converter"

# 양식 선택:
#   "auto" = 자동 감지 (권장)
#   "old"  = 2023-2024년 양식 (페이지별 폼)
#   "new"  = 2025-2026년 양식 (표 형태)
FORMAT = "auto"

# 추출 엔진 선택 (new 양식에만 적용):
#   "pdfplumber" 또는 "tabula"
ENGINE = "pdfplumber"

#===================================================

## 3단계: 변환 실행
아래 셀을 실행하면 자동으로 변환됩니다.

In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

# 통합 컬럼명
STANDARD_COLUMNS = [
    "날짜", "차량번호", "운전자", "소속", "사용목적", "행선지",
    "출발시간", "도착시간", "출발(km)", "도착(km)", "주행거리(km)",
    "수령량(ℓ)", "동승자", "비고",
]


# ========== 양식 자동 감지 ==========
def detect_format(pdf_path):
    import pdfplumber
    with pdfplumber.open(pdf_path) as pdf:
        if not pdf.pages:
            return "new"
        text = pdf.pages[0].extract_text() or ""
        old_markers = ["차량운행일지", "계기표시", "전일누계", "금일주행", "관리운전원"]
        matches = sum(1 for m in old_markers if m in text.replace(" ", ""))
        return "old" if matches >= 2 else "new"


# ========== 구 양식 (2023-2024) 파서 ==========
def parse_old_format(pdf_path):
    import pdfplumber
    records = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            text = page.extract_text() or ""
            tables = page.extract_tables() or []
            if not text.strip():
                continue
            result = _parse_old_page(text, tables)
            if result:
                if isinstance(result, list):
                    records.extend(result)
                else:
                    records.append(result)
    if not records:
        return pd.DataFrame(columns=STANDARD_COLUMNS)
    return pd.DataFrame(records, columns=STANDARD_COLUMNS)


def _parse_old_page(text, tables):
    lines = text.split("\n")
    flat = text.replace(" ", "")

    # 날짜
    date_str = ""
    dm = re.search(r"(\d{4})\s*년\s*(\d{1,2})\s*월\s*(\d{1,2})\s*일", text)
    if dm:
        date_str = f"{dm.group(1)}-{int(dm.group(2)):02d}-{int(dm.group(3)):02d}"

    # 차량번호
    vehicle_no = ""
    vm = re.search(r"차량번호\s*(\S+)", text)
    if vm:
        vehicle_no = vm.group(1)

    # 사용목적
    purpose = ""
    pm = re.search(r"사용목적\s+(.+)", text)
    if pm:
        purpose = pm.group(1).strip()

    # 운행 건 추출 (테이블에서)
    trips = []
    for table in tables:
        for row in table:
            if not row or len(row) < 3:
                continue
            cells = [str(c).strip() if c else "" for c in row]
            joined = "".join(cells)
            if re.search(r"출\s*[:\uff1a]?\s*\d{1,2}[:\uff1a]\d{2}", joined):
                trip = _extract_trip(cells, joined)
                if trip:
                    trips.append(trip)

    # 텍스트 폴백
    if not trips:
        for line in lines:
            dep = re.search(r"출\s*[:\uff1a]?\s*(\d{1,2}[:\uff1a]\d{2})", line)
            if dep:
                trip = {"depart_time": dep.group(1).replace("\uff1a", ":")}
                arr = re.search(r"착\s*[:\uff1a]?\s*(\d{1,2}[:\uff1a]\d{2})", line)
                if arr:
                    trip["arrive_time"] = arr.group(1).replace("\uff1a", ":")
                nums = re.findall(r"\d{4,6}", line)
                if len(nums) >= 2:
                    trip["start_km"], trip["end_km"] = nums[0], nums[1]
                names = re.findall(r"[가-힣]{2,4}", line)
                skip = {"출발", "도착", "운행", "시간", "계기", "표시", "운전", "동승", "소속", "성명"}
                for n in names:
                    if n not in skip:
                        trip["driver"] = n
                        break
                trips.append(trip)

    # 주행거리
    prev_km = ""
    today_km = ""
    curr_km = ""
    m = re.search(r"전일누계\s*(\d[\d,]*)\s*km", text, re.I)
    if m: prev_km = m.group(1).replace(",", "")
    m = re.search(r"금일주행\s*(\d[\d,]*)\s*km", text, re.I)
    if m: today_km = m.group(1).replace(",", "")
    m = re.search(r"금일누계\s*(\d[\d,]*)\s*km", text, re.I)
    if m: curr_km = m.group(1).replace(",", "")

    # 수령량
    fuel = ""
    fm = re.search(r"수령량\s*(\d[\d.]*)\s*[ℓlL]", text)
    if fm: fuel = fm.group(1)

    if trips:
        records = []
        for i, t in enumerate(trips):
            sk = t.get("start_km", "")
            ek = t.get("end_km", "")
            dist = t.get("distance", "")
            if not sk and i == 0 and prev_km: sk = prev_km
            if not ek and i == len(trips)-1 and curr_km: ek = curr_km
            if not dist and sk and ek:
                try: dist = str(int(ek.replace(",","")) - int(sk.replace(",","")))
                except: pass
            records.append({
                "날짜": date_str, "차량번호": vehicle_no,
                "운전자": t.get("driver",""), "소속": t.get("department",""),
                "사용목적": purpose, "행선지": t.get("destination",""),
                "출발시간": t.get("depart_time",""), "도착시간": t.get("arrive_time",""),
                "출발(km)": sk, "도착(km)": ek,
                "주행거리(km)": dist if dist else today_km,
                "수령량(ℓ)": fuel if i==0 else "",
                "동승자": t.get("passenger",""), "비고": "",
            })
        return records

    if not date_str and not vehicle_no:
        return None
    dist = today_km
    if not dist and prev_km and curr_km:
        try: dist = str(int(curr_km) - int(prev_km))
        except: pass
    return {
        "날짜": date_str, "차량번호": vehicle_no,
        "운전자": "", "소속": "", "사용목적": purpose, "행선지": "",
        "출발시간": "", "도착시간": "",
        "출발(km)": prev_km, "도착(km)": curr_km,
        "주행거리(km)": dist, "수령량(ℓ)": fuel,
        "동승자": "", "비고": "",
    }


def _extract_trip(cells, joined):
    trip = {}
    dep = re.search(r"출\s*[:\uff1a]?\s*(\d{1,2}[:\uff1a]\d{2})", joined)
    if dep: trip["depart_time"] = dep.group(1).replace("\uff1a", ":")
    arr = re.search(r"착\s*[:\uff1a]?\s*(\d{1,2}[:\uff1a]\d{2})", joined)
    if arr: trip["arrive_time"] = arr.group(1).replace("\uff1a", ":")
    nums = re.findall(r"\d{4,6}", joined)
    if len(nums) >= 2:
        trip["start_km"], trip["end_km"] = nums[0], nums[1]
    for c in cells:
        if c and re.search(r"[가-힣]", c) and "출" not in c and "착" not in c:
            if not any(kw in c for kw in ["계기","운행","운전","동승","소속"]):
                trip["destination"] = c
                break
    for c in cells:
        if c and re.fullmatch(r"[가-힣]{2,4}", c):
            trip["driver"] = c
            break
    for c in cells:
        if c and ("팀" in c or "부" in c or "실" in c):
            trip["department"] = c
            break
    return trip if trip.get("depart_time") else None


# ========== 신 양식 (2025-2026) 파서 ==========
def extract_tables_pdfplumber(pdf_path):
    import pdfplumber
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for table in (page.extract_tables() or []):
                if table:
                    cleaned = [r for r in table if any(c and str(c).strip() for c in r)]
                    if cleaned:
                        tables.append(pd.DataFrame(cleaned))
    return tables


def extract_tables_tabula(pdf_path):
    import tabula
    try:
        tables = tabula.read_pdf(pdf_path, pages="all", multiple_tables=True, lattice=True)
    except: tables = []
    if not tables:
        try:
            tables = tabula.read_pdf(pdf_path, pages="all", multiple_tables=True, stream=True)
        except: tables = []
    return [t for t in tables if not t.empty]


def promote_header(df):
    if df.empty: return df
    first_row = df.iloc[0]
    header_like = sum(1 for v in first_row if v and str(v).strip() and not str(v).strip().replace(".","").isdigit())
    if header_like >= len(first_row) * 0.5:
        headers = [str(v).strip() if v else f"Col_{i}" for i, v in enumerate(first_row)]
        df = df.iloc[1:].reset_index(drop=True)
        df.columns = headers
    return df


def merge_tables(tables):
    if not tables: return pd.DataFrame()
    if len(tables) == 1: return promote_header(tables[0])
    groups = {}
    for t in tables:
        groups.setdefault(len(t.columns), []).append(t)
    largest = max(groups.values(), key=lambda g: sum(len(t) for t in g))
    norm = []
    for t in largest:
        t = t.copy(); t.columns = range(len(t.columns)); norm.append(t)
    return promote_header(pd.concat(norm, ignore_index=True))


# ========== Excel 스타일링 ==========
def style_excel(wb_path):
    wb = load_workbook(wb_path)
    ws = wb.active
    hf = Font(name="맑은 고딕", bold=True, size=11, color="FFFFFF")
    hfill = PatternFill(start_color="2F5496", end_color="2F5496", fill_type="solid")
    cf = Font(name="맑은 고딕", size=10)
    border = Border(left=Side(style="thin"), right=Side(style="thin"),
                    top=Side(style="thin"), bottom=Side(style="thin"))
    ca = Alignment(horizontal="center", vertical="center", wrap_text=True)
    la = Alignment(horizontal="left", vertical="center", wrap_text=True)
    for ri, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, max_col=ws.max_column), 1):
        for cell in row:
            cell.border = border
            if ri == 1: cell.font = hf; cell.fill = hfill; cell.alignment = ca
            else: cell.font = cf; cell.alignment = la
    for ci in range(1, ws.max_column + 1):
        ml = 0; cl = get_column_letter(ci)
        for cell in ws[cl]:
            if cell.value:
                ml = max(ml, sum(2 if ord(c) > 127 else 1 for c in str(cell.value)))
        ws.column_dimensions[cl].width = min(max(ml + 4, 8), 50)
    ws.freeze_panes = "A2"
    wb.save(wb_path)


# ========== 통합 변환 함수 ==========
def convert_pdf_to_excel(pdf_path, output_path=None, engine="pdfplumber", pdf_format="auto"):
    pdf_path = str(Path(pdf_path).resolve())
    if output_path is None:
        output_path = str(Path(pdf_path).with_suffix(".xlsx"))

    if pdf_format == "auto":
        pdf_format = detect_format(pdf_path)

    fmt_label = "2023-2024 양식(폼)" if pdf_format == "old" else "2025-2026 양식(표)"
    print(f"  감지된 양식: {fmt_label}")

    if pdf_format == "old":
        df = parse_old_format(pdf_path)
        if df.empty:
            print(f"  경고: 데이터를 찾지 못했습니다.")
            return None
        print(f"  추출된 레코드: {len(df)}건")
    else:
        if engine == "pdfplumber":
            tables = extract_tables_pdfplumber(pdf_path)
        else:
            tables = extract_tables_tabula(pdf_path)
        if not tables:
            print(f"  경고: 테이블을 찾지 못했습니다.")
            return None
        df = merge_tables(tables)
        print(f"  추출된 행: {len(df)}건")

    df.to_excel(output_path, index=False, sheet_name="운행일지")
    style_excel(output_path)
    return output_path


print("변환 함수 준비 완료! (구/신 양식 모두 지원)")

In [ ]:
# 변환 실행
input_path = Path(PDF_PATH)

if input_path.is_dir():
    pdf_files = sorted(input_path.glob("*.pdf")) + sorted(input_path.glob("*.PDF"))
    print(f"폴더에서 PDF {len(pdf_files)}개 발견\n")
    results = []
    for i, pdf_file in enumerate(pdf_files, 1):
        print(f"[{i}/{len(pdf_files)}] {pdf_file.name} 변환 중...")
        result = convert_pdf_to_excel(str(pdf_file), engine=ENGINE, pdf_format=FORMAT)
        if result:
            results.append(result)
            print(f"  → {Path(result).name} 완료!\n")
    print(f"===== 총 {len(results)}개 파일 변환 완료! =====")
elif input_path.is_file():
    print(f"{input_path.name} 변환 중...")
    result = convert_pdf_to_excel(str(input_path), engine=ENGINE, pdf_format=FORMAT)
    if result:
        print(f"\n===== 완료! → {result} =====")
else:
    print(f"경로를 찾을 수 없습니다: {PDF_PATH}")
    print("2단계에서 경로를 다시 확인해주세요.")

## 4단계: 결과 미리보기 (선택사항)
변환된 첫 번째 엑셀 파일의 내용을 확인합니다.

In [ ]:
# 변환된 엑셀 파일 미리보기
if input_path.is_dir():
    xlsx_files = sorted(input_path.glob("*.xlsx"))
    if xlsx_files:
        print(f"미리보기: {xlsx_files[0].name}\n")
        df = pd.read_excel(xlsx_files[0])
        display(df)
    else:
        print("변환된 xlsx 파일이 없습니다.")
elif input_path.is_file():
    xlsx_path = input_path.with_suffix(".xlsx")
    if xlsx_path.exists():
        print(f"미리보기: {xlsx_path.name}\n")
        df = pd.read_excel(xlsx_path)
        display(df)